<a href="https://colab.research.google.com/github/6hamuge/Beyond-ETRI/blob/main/ptdt_xgboost_enhanced_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost LOSO-CV
데이터 로딩 → 피처 엔지니어링 → 도메인 파생 피처 → XGBoost LOSO-CV → 제출 파일 생성

**추가 피처 (기본 센서 피처 위에 쌓임)**

| 피처 | 타겟 레이블 |
|------|------------|
| Lag-1 수면/활동 | Q1, S1~S4 |
| Sleep debt 3d/7d | Q1, Q2, S1 |
| 수면 타이밍 불규칙성 | Q1, S3 |
| Pre-sleep arousal index | Q3, S3 |
| Activity-recovery ratio | Q2, S2 |
| 연속 수면 부족 streak | Q1, Q2 |
| Social jet lag | Q1, S3 |
| Screen-sleep delay | S3, Q1 |

## 0. 패키지 설치

In [ ]:
!pip install pyarrow xgboost scikit-learn -q
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import QuantileTransformer
from functools import reduce
import matplotlib.pyplot as plt, matplotlib, os, warnings, math
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

LABEL_COLS = ['Q1','Q2','Q3','S1','S2','S3','S4']
Q_COLS, S_COLS = ['Q1','Q2','Q3'], ['S1','S2','S3','S4']
print(f'XGBoost {xgb.__version__}  ✅')

XGBoost 3.2.0  ✅


## 1. 데이터 로딩

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ch2025/'

FILE_MAP = {
    'ac_status'  : 'ch2025_mACStatus.parquet',
    'activity'   : 'ch2025_mActivity.parquet',
    'ambience'   : 'ch2025_mAmbience.parquet',
    'ble'        : 'ch2025_mBle.parquet',
    'gps'        : 'ch2025_mGps.parquet',
    'hr'         : 'ch2025_mHr.parquet',
    'light'      : 'ch2025_mLight.parquet',
    'pedo'       : 'ch2025_mPedometer.parquet',
    'screen'     : 'ch2025_mScreenStatus.parquet',
    'usage_stats': 'ch2025_mUsageStats.parquet',
    'w_light'    : 'ch2025_wLight.parquet',
    'wifi'       : 'ch2025_mWifi.parquet',
}
raw = {}
for key, fname in FILE_MAP.items():
    path = os.path.join(DATA_DIR,"ch2025_data_items/", fname)
    if os.path.exists(path):
        raw[key] = pd.read_parquet(path)
        print(f'  ✅ [{key}]: {raw[key].shape}')
    else:
        print(f'  ⚠️  [{key}] 없음')
print(f'\n총 {len(raw)}개 센서 로드')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅ [ac_status]: (939896, 3)
  ✅ [activity]: (961062, 3)
  ✅ [ambience]: (476577, 3)
  ✅ [ble]: (21830, 3)
  ✅ [gps]: (800611, 3)
  ⚠️  [hr] 없음
  ✅ [light]: (96258, 3)
  ⚠️  [pedo] 없음
  ✅ [screen]: (939653, 3)
  ✅ [usage_stats]: (45197, 3)
  ✅ [w_light]: (633741, 3)
  ✅ [wifi]: (76336, 3)

총 10개 센서 로드


## 2. 기본 피처 엔지니어링
### 2-A. 일별 집계 피처

In [ ]:
def get_timestamp_col(df):
    for c in df.columns:
        if any(k in c.lower() for k in ['timestamp','time','datetime','ts','date']):
            return c
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]): return c
    return None

def extract_daily_features(df, sensor_name, subject_col='subject_id'):
    df = df.copy()
    ts_col = get_timestamp_col(df)
    if ts_col and ts_col in df.columns:
        df[ts_col] = (pd.to_datetime(df[ts_col], unit='ms', errors='coerce')
                      if df[ts_col].dtype in ['int64','float64']
                      else pd.to_datetime(df[ts_col], errors='coerce'))
        df['date'] = df[ts_col].dt.date
    elif 'date' not in df.columns:
        print(f'  ⚠️  [{sensor_name}] 타임스탬프 없음'); return None
    df['date'] = pd.to_datetime(df['date'])
    exclude  = {subject_col,'date',ts_col}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
    if not num_cols: return None
    grp = df.groupby([subject_col,'date'])[num_cols]
    parts = []
    for agg_name, agg_df in {'mean':grp.mean(),'std':grp.std().fillna(0),
                              'min':grp.min(),'max':grp.max(),'count':grp.count()}.items():
        agg_df.columns = [f'{sensor_name}__{c}__{agg_name}' for c in agg_df.columns]
        parts.append(agg_df)
    result = pd.concat(parts, axis=1).reset_index()
    print(f'  ✅ [{sensor_name}]: {result.shape[1]-2}개 피처')
    return result

print('📊 일별 피처 추출...')
daily_features = {}
for key, df in raw.items():
    feat = extract_daily_features(df, sensor_name=key)
    if feat is not None: daily_features[key] = feat
print(f'\n✅ 기본 피처: {len(daily_features)}개 센서')

📊 일별 피처 추출...
  ✅ [ac_status]: 5개 피처
  ✅ [activity]: 5개 피처
  ✅ [light]: 5개 피처
  ✅ [screen]: 5개 피처
  ✅ [w_light]: 5개 피처

✅ 기본 피처: 5개 센서


### 2-B. 중첩 리스트 센서 피처

In [ ]:
def extract_hr_features_nested(df, subject_col='subject_id'):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = pd.to_datetime(df['timestamp'].dt.date)
    df['hour'] = df['timestamp'].dt.hour
    df = df.explode('heart_rate')
    df['heart_rate'] = pd.to_numeric(df['heart_rate'], errors='coerce')
    df = df.dropna(subset=['heart_rate'])
    df = df[df['heart_rate'].between(30, 220)]
    results = []
    for (subj, date), grp in df.groupby([subject_col, 'date']):
        hr = grp['heart_rate']
        night = grp[grp['hour'].between(0,6)]['heart_rate']
        pre   = grp[grp['hour'].between(21,23)]['heart_rate']
        results.append({subject_col:subj,'date':date,
            'hr__mean':hr.mean(),'hr__std':hr.std(ddof=0),
            'hr__min':hr.min(),'hr__max':hr.max(),'hr__count':len(hr),
            'hr__resting':hr.quantile(0.1),
            'hr__rmssd':float(np.sqrt(np.mean(np.diff(hr.values)**2))) if len(hr)>1 else 0,
            'hr__night_mean':night.mean() if len(night)>0 else np.nan,
            'hr__night_std':night.std(ddof=0) if len(night)>1 else 0,
            'hr__night_min':night.min() if len(night)>0 else np.nan,
            'hr__presleep_mean':pre.mean() if len(pre)>0 else np.nan})
    result_df = pd.DataFrame(results)
    print(f'  ✅ [hr]: {result_df.shape[1]-2}개 피처')
    return result_df

def extract_wifi_features_nested(df, subject_col='subject_id'):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = pd.to_datetime(df['timestamp'].dt.date)
    df['hour'] = df['timestamp'].dt.hour
    df = df.explode('m_wifi').dropna(subset=['m_wifi'])
    wifi_exp = pd.json_normalize(df['m_wifi'].tolist())
    df = pd.concat([df[[subject_col,'date','hour','timestamp']].reset_index(drop=True),
                    wifi_exp.reset_index(drop=True)], axis=1)
    if 'rssi' in df.columns: df['rssi'] = pd.to_numeric(df['rssi'], errors='coerce')
    home_bssid = (df.groupby(subject_col)['bssid']
                    .agg(lambda x: x.value_counts().index[0]).to_dict())                   if 'bssid' in df.columns else {}
    if home_bssid:
        df['is_home'] = df.apply(lambda r: int(r['bssid']==home_bssid.get(r[subject_col],'')), axis=1)
    results = []
    for (subj,date),grp in df.groupby([subject_col,'date']):
        night = grp[grp['hour'].between(0,6)]
        row = {subject_col:subj,'date':date,
               'wifi__scan_count':len(grp),
               'wifi__unique_bssid':grp['bssid'].nunique() if 'bssid' in grp else 0,
               'wifi__night_scan_count':len(night),
               'wifi__home_ratio':grp['is_home'].mean() if 'is_home' in grp else 0,
               'wifi__home_count':grp['is_home'].sum() if 'is_home' in grp else 0}
        if 'rssi' in grp.columns and grp['rssi'].notna().sum()>0:
            row['wifi__rssi_mean']=grp['rssi'].mean(); row['wifi__rssi_std']=grp['rssi'].std(ddof=0)
        if 'is_home' in grp.columns:
            eve=grp[(grp['is_home']==1)&(grp['hour']>=18)]
            row['wifi__home_arrival_hour']=eve['hour'].min() if len(eve)>0 else np.nan
        results.append(row)
    result_df = pd.DataFrame(results)
    print(f'  ✅ [wifi]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_usage_stats_features_nested(df, subject_col='subject_id'):
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
    df['date'] = pd.to_datetime(df['timestamp'].dt.date)
    df['hour'] = df['timestamp'].dt.hour
    df = df.explode('m_usage_stats').dropna(subset=['m_usage_stats'])
    usage_exp = pd.json_normalize(df['m_usage_stats'].tolist())
    df = pd.concat([df[[subject_col,'date','hour']].reset_index(drop=True),
                    usage_exp.reset_index(drop=True)], axis=1)
    if 'total_time' in df.columns:
        df['total_time'] = pd.to_numeric(df['total_time'], errors='coerce').fillna(0)
        df['total_time_min'] = df['total_time']/60000
    CATS = {'social':['kakao','instagram','facebook','twitter','tiktok','line'],
            'video':['youtube','netflix','tving','watcha'],'game':['game','pubg','minecraft'],
            'browser':['chrome','samsung.internet','naver'],'work':['office','notion','slack','zoom']}
    if 'app_name' in df.columns:
        df['category']='other'
        for cat,kws in CATS.items():
            df.loc[df['app_name'].str.lower().str.contains('|'.join(kws),na=False),'category']=cat
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        row={subject_col:subj,'date':date,
             'usage__app_count':grp['app_name'].nunique() if 'app_name' in grp else 0}
        if 'total_time_min' in grp.columns:
            total=grp['total_time_min'].sum(); row['usage__total_min']=total
            eve=grp[grp['hour'].between(22,23)]['total_time_min'].sum()
            row['usage__evening_min']=eve; row['usage__evening_ratio']=eve/(total+1e-6)
            for cat in CATS:
                t=grp[grp['category']==cat]['total_time_min'].sum()
                row[f'usage__{cat}_min']=t; row[f'usage__{cat}_ratio']=t/(total+1e-6)
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [usage_stats]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_ble_features_nested(df, subject_col='subject_id'):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df['date']=pd.to_datetime(df['timestamp'].dt.date); df['hour']=df['timestamp'].dt.hour
    df=df.explode('m_ble').dropna(subset=['m_ble'])
    ble_exp=pd.json_normalize(df['m_ble'].tolist())
    df=pd.concat([df[[subject_col,'date','hour']].reset_index(drop=True),
                  ble_exp.reset_index(drop=True)],axis=1)
    if 'rssi' in df.columns: df['rssi']=pd.to_numeric(df['rssi'],errors='coerce')
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        night=grp[grp['hour'].between(0,6)]
        row={subject_col:subj,'date':date,'ble__scan_count':len(grp),
             'ble__unique_devices':grp['address'].nunique() if 'address' in grp else 0,
             'ble__night_scan_count':len(night),
             'ble__night_unique_devs':night['address'].nunique()
              if 'address' in night.columns and len(night)>0 else 0}
        if 'rssi' in grp.columns and grp['rssi'].notna().sum()>0:
            row['ble__rssi_mean']=grp['rssi'].mean(); row['ble__rssi_max']=grp['rssi'].max()
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [ble]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_gps_features_nested(df, subject_col='subject_id'):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df['date']=pd.to_datetime(df['timestamp'].dt.date); df['hour']=df['timestamp'].dt.hour
    df=df.explode('m_gps').dropna(subset=['m_gps'])
    gps_exp=pd.json_normalize(df['m_gps'].tolist())
    df=pd.concat([df[[subject_col,'date','hour']].reset_index(drop=True),
                  gps_exp.reset_index(drop=True)],axis=1)
    for c in ['latitude','longitude','speed','altitude']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        row={subject_col:subj,'date':date,'gps__fix_count':len(grp)}
        if 'latitude' in grp.columns and grp['latitude'].notna().sum()>1:
            row['gps__lat_std']=grp['latitude'].std(); row['gps__lon_std']=grp['longitude'].std()
            row['gps__radius']=np.sqrt(grp['latitude'].std()**2+grp['longitude'].std()**2)
        if 'speed' in grp.columns and grp['speed'].notna().sum()>0:
            row['gps__speed_mean']=grp['speed'].mean(); row['gps__speed_max']=grp['speed'].max()
            row['gps__moving_ratio']=(grp['speed']>0.5).mean()
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [gps]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_ambience_features_nested(df, subject_col='subject_id'):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df['date']=pd.to_datetime(df['timestamp'].dt.date)
    def flatten(val):
        if val is None: return []
        if isinstance(val,list):
            r=[]
            for item in val: r.extend(item if isinstance(item,list) else [str(item)])
            return r
        return [str(val)]
    df['labels']=df['m_ambience'].apply(flatten)
    df=df.explode('labels').dropna(subset=['labels'])
    df['labels']=df['labels'].astype(str).str.strip().str.lower()
    df=df[df['labels']!='']
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        total=len(grp); vc=grp['labels'].value_counts()
        row={subject_col:subj,'date':date,'ambience__total_count':total,
             'ambience__unique_labels':grp['labels'].nunique()}
        for lbl in vc.index[:10]:
            row[f'ambience__{lbl.replace(" ","_").replace("/","_")}_ratio']=vc[lbl]/total
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [ambience]: {result_df.shape[1]-2}개 피처'); return result_df

print('📊 중첩 센서 피처 추출...')
NESTED = {'hr':extract_hr_features_nested,'wifi':extract_wifi_features_nested,
          'usage_stats':extract_usage_stats_features_nested,'ble':extract_ble_features_nested,
          'gps':extract_gps_features_nested,'ambience':extract_ambience_features_nested}
for key, func in NESTED.items():
    if key in raw:
        feat = func(raw[key])
        if feat is not None: daily_features[key] = feat
print(f'\n✅ 완료: {list(daily_features.keys())}')

📊 중첩 센서 피처 추출...
  ✅ [wifi]: 8개 피처
  ✅ [usage_stats]: 14개 피처
  ✅ [ble]: 6개 피처
  ✅ [gps]: 7개 피처
  ✅ [ambience]: 4986개 피처

✅ 완료: ['ac_status', 'activity', 'light', 'screen', 'w_light', 'wifi', 'usage_stats', 'ble', 'gps', 'ambience']


### 2-C. 특화 피처 (수면 구간 추정 / WiFi·UsageStats 취침 프록시)

In [ ]:
def ensure_datetime(df, ts_col='timestamp'):
    df=df.copy(); df[ts_col]=pd.to_datetime(df[ts_col],errors='coerce')
    df['date']=pd.to_datetime(df[ts_col].dt.date); df['hour']=df[ts_col].dt.hour
    return df

def to_list(x):
    if x is None: return []
    if isinstance(x,(np.ndarray,list)): return list(x)
    if isinstance(x,dict): return [x]
    return []

def estimate_sleep_from_hr(df, subject_col='subject_id', min_sleep_duration_min=60):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df=df.explode('heart_rate'); df['heart_rate']=pd.to_numeric(df['heart_rate'],errors='coerce')
    df=df.dropna(subset=['heart_rate','timestamp'])
    df=df[df['heart_rate'].between(30,220)].sort_values([subject_col,'timestamp'])
    results=[]
    for subj,subj_df in df.groupby(subject_col):
        sleep_thr=subj_df['heart_rate'].quantile(0.35)
        for date in subj_df['timestamp'].dt.date.unique():
            date=pd.Timestamp(date)
            win=subj_df[(subj_df['timestamp']>=date-pd.Timedelta(hours=4))&
                        (subj_df['timestamp']<=date+pd.Timedelta(hours=12))].copy()
            row={subject_col:subj,'date':date}
            SLEEP_COLS=['hr__sleep_onset_hour','hr__sleep_offset_hour','hr__est_tst_min',
                        'hr__sleep_hr_mean','hr__sleep_hr_std','hr__sleep_hr_min',
                        'hr__presleep_hr_drop','hr__arousal_count']
            if len(win)<10:
                for c in SLEEP_COLS: row[c]=np.nan
                results.append(row); continue
            win1m=win.set_index('timestamp')['heart_rate'].resample('1min').mean().interpolate()
            is_low=win1m<sleep_thr
            onset=offset=None; max_dur=0; in_sleep=False; seg=None
            for t,low in is_low.items():
                if low and not in_sleep: in_sleep=True; seg=t
                elif not low and in_sleep:
                    dur=(t-seg).total_seconds()/60
                    if dur>max_dur and dur>=min_sleep_duration_min:
                        max_dur=dur; onset=seg; offset=t
                    in_sleep=False
            if onset and offset:
                slp=win1m[onset:offset]
                pre=win1m[max(win1m.index[0],onset-pd.Timedelta(hours=1)):onset]
                row.update({'hr__sleep_onset_hour':onset.hour+onset.minute/60,
                            'hr__sleep_offset_hour':offset.hour+offset.minute/60,
                            'hr__est_tst_min':max_dur,'hr__sleep_hr_mean':slp.mean(),
                            'hr__sleep_hr_std':slp.std(),'hr__sleep_hr_min':slp.min(),
                            'hr__presleep_hr_drop':pre.mean()-slp.mean() if len(pre)>0 else np.nan,
                            'hr__arousal_count':int((slp>sleep_thr).sum())})
            else:
                for c in SLEEP_COLS: row[c]=np.nan
            results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [hr_sleep]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_wifi_bedtime_proxy(df, subject_col='subject_id'):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df['date']=pd.to_datetime(df['timestamp'].dt.date); df['hour']=df['timestamp'].dt.hour
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        night=grp[grp['hour'].between(20,23)|grp['hour'].between(0,6)].sort_values('timestamp')
        row={subject_col:subj,'date':date,'wifi__phone_down_hour':np.nan,
             'wifi__last_night_scan_hour':np.nan,'wifi__first_morning_scan_hour':np.nan}
        if len(night)>=2:
            night=night.set_index('timestamp')
            gaps=night.index.to_series().diff().dt.total_seconds()/60
            long_gap=gaps[gaps>30]
            if len(long_gap)>0:
                gs=long_gap.index[0]-pd.Timedelta(minutes=gaps[long_gap.index[0]])
                row['wifi__phone_down_hour']=gs.hour+gs.minute/60
            row['wifi__last_night_scan_hour']=night.index[-1].hour+night.index[-1].minute/60
            morn=grp[grp['hour'].between(5,9)].sort_values('timestamp')
            if len(morn)>0:
                f=morn['timestamp'].iloc[0]
                row['wifi__first_morning_scan_hour']=f.hour+f.minute/60
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [wifi_bedtime]: {result_df.shape[1]-2}개 피처'); return result_df

def extract_usage_bedtime_proxy(df, subject_col='subject_id'):
    df=df.copy()
    df['timestamp']=pd.to_datetime(df['timestamp'],unit='ms',errors='coerce')
    df['date']=pd.to_datetime(df['timestamp'].dt.date); df['hour']=df['timestamp'].dt.hour
    results=[]
    for (subj,date),grp in df.groupby([subject_col,'date']):
        row={subject_col:subj,'date':date}
        eve=grp[grp['hour']>=20].sort_values('timestamp')
        row['usage__last_use_hour']=eve['timestamp'].iloc[-1].hour+eve['timestamp'].iloc[-1].minute/60 if len(eve)>0 else np.nan
        morn=grp[grp['hour'].between(5,10)].sort_values('timestamp')
        row['usage__first_morning_hour']=morn['timestamp'].iloc[0].hour+morn['timestamp'].iloc[0].minute/60 if len(morn)>0 else np.nan
        row['usage__sleep_hour_count']=(grp['hour'].between(0,6)).sum()
        if not pd.isna(row.get('usage__last_use_hour')) and not pd.isna(row.get('usage__first_morning_hour')):
            on=row['usage__last_use_hour']; off=row['usage__first_morning_hour']
            row['usage__phone_off_duration_hr']=(off-on) if off>on else (24-on+off)
        results.append(row)
    result_df=pd.DataFrame(results)
    print(f'  ✅ [usage_bedtime]: {result_df.shape[1]-2}개 피처'); return result_df

print('📊 수면 시각 추정 피처...')
if 'hr' in raw:
    hr_sleep=estimate_sleep_from_hr(raw['hr'])
    daily_features['hr']=pd.merge(daily_features['hr'],hr_sleep,on=['subject_id','date'],how='outer') if 'hr' in daily_features else hr_sleep
if 'wifi' in raw:
    wifi_bed=extract_wifi_bedtime_proxy(raw['wifi'])
    if 'wifi' in daily_features:
        daily_features['wifi']=pd.merge(daily_features['wifi'],wifi_bed,on=['subject_id','date'],how='outer')
if 'usage_stats' in raw:
    usage_bed=extract_usage_bedtime_proxy(raw['usage_stats'])
    if 'usage_stats' in daily_features:
        daily_features['usage_stats']=pd.merge(daily_features['usage_stats'],usage_bed,on=['subject_id','date'],how='outer')
print('✅ 완료')

### 2-D. 피처 병합 + 시간 주기 피처

In [ ]:
feature_df = reduce(lambda l,r: pd.merge(l,r,on=['subject_id','date'],how='outer'),
                    daily_features.values())
feature_df['date'] = pd.to_datetime(feature_df['date'])

dow = feature_df['date'].dt.dayofweek
feature_df['time__dow_sin']    = np.sin(2*np.pi*dow/7)
feature_df['time__dow_cos']    = np.cos(2*np.pi*dow/7)
feature_df['time__is_weekend'] = (dow>=5).astype(int)
month = feature_df['date'].dt.month
feature_df['time__month_sin']  = np.sin(2*np.pi*month/12)
feature_df['time__month_cos']  = np.cos(2*np.pi*month/12)

# consensus 수면 시각
def add_consensus(df):
    def _c(row):
        vals,ws=[],[]
        for col,w in [('hr__sleep_onset_hour',0.6),('wifi__phone_down_hour',0.25),('usage__last_use_hour',0.15)]:
            if col in df.columns and pd.notna(row.get(col)):
                vals.append(row[col]); ws.append(w)
        return np.average(vals,weights=ws[:len(vals)]) if vals else np.nan
    df['sleep__consensus_onset_hour']=df.apply(_c,axis=1); return df

feature_df = add_consensus(feature_df)
print(f'✅ 통합 피처: {feature_df.shape}  |  피처 수: {feature_df.shape[1]-2}')

### 2-E. 도메인 시간대별 피처 (주간 / 취침 전 / 야간)

In [ ]:
DAYTIME=(8,21); PRESLEEP=(21,24); SLEEP_NIGHT=(0,7)

def _parse_ts(df):
    df=df.copy()
    ts_col=next((c for c in df.columns if any(k in c.lower() for k in ['timestamp','time','ts','datetime'])),None)
    if ts_col is None: return None
    df['_dt']=(pd.to_datetime(df[ts_col],unit='ms',errors='coerce')
               if df[ts_col].dtype in ['int64','float64']
               else pd.to_datetime(df[ts_col],errors='coerce'))
    df['date']=df['_dt'].dt.normalize(); df['hour']=df['_dt'].dt.hour; return df

def _wm(df,h0,h1):
    return (df['hour']>=h0)&(df['hour']<h1) if h1>=h0 else (df['hour']>=h0)|(df['hour']<h1)

def _feat_hr(df):
    df=_parse_ts(df)
    if df is None: return pd.DataFrame()
    hc=next((c for c in df.columns if 'heart' in c.lower() or c.lower() in ['hr','bpm']),None)
    if hc is None: return pd.DataFrame()
    rows=[]
    for (subj,date),g in df.groupby(['subject_id','date']):
        r={'subject_id':subj,'date':date}
        day=g[_wm(g,*DAYTIME)][hc].dropna(); pre=g[_wm(g,*PRESLEEP)][hc].dropna()
        night=g[_wm(g,*SLEEP_NIGHT)][hc].dropna()
        r['hr_day_mean']=day.mean(); r['hr_day_max']=day.max(); r['hr_day_above100']=(day>100).mean()
        r['hr_presleep_mean']=pre.mean()
        r['hr_presleep_trend']=pre.iloc[-len(pre)//3:].mean()-pre.iloc[:len(pre)//3].mean() if len(pre)>=6 else np.nan
        r['hr_night_mean']=night.mean(); r['hr_night_std']=night.std(); r['hr_night_min']=night.min()
        r['hr_night_sleep_frac']=(night<60).mean() if len(night)>0 else np.nan
        if len(night)>10:
            ab=(night>75).astype(int); r['hr_night_spikes']=(ab.diff().fillna(0)==1).sum()
        else: r['hr_night_spikes']=np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def _feat_screen(df):
    df=_parse_ts(df)
    if df is None: return pd.DataFrame()
    sc=next((c for c in df.columns if any(k in c.lower() for k in ['status','screen','state'])),None)
    if sc is None: return pd.DataFrame()
    rows=[]
    for (subj,date),g in df.groupby(['subject_id','date']):
        r={'subject_id':subj,'date':date}; g=g.sort_values('_dt')
        pre=g[_wm(g,*PRESLEEP)]; night=g[_wm(g,*SLEEP_NIGHT)]
        if len(pre)>0:
            on=(pre[sc]==1); r['screen_presleep_on_count']=on.sum(); r['screen_presleep_on_frac']=on.mean()
            off_t=pre[pre[sc]==0]['_dt']
            r['screen_last_off_hour']=off_t.iloc[-1].hour+off_t.iloc[-1].minute/60 if len(off_t)>0 else 21.0
        if len(night)>0:
            r['screen_night_off_frac']=(night[sc]==0).mean()
            r['screen_night_on_count']=((night[sc]==1).astype(int).diff().fillna(0)==1).sum()
        r['screen_total_on_count']=(g[sc]==1).sum()
        rows.append(r)
    return pd.DataFrame(rows)

def _feat_pedo(df):
    df=_parse_ts(df)
    if df is None: return pd.DataFrame()
    sc=next((c for c in df.columns if 'step' in c.lower()),None)
    cal=next((c for c in df.columns if 'cal' in c.lower()),None)
    rows=[]
    for (subj,date),g in df.groupby(['subject_id','date']):
        r={'subject_id':subj,'date':date}
        if sc:
            ds=g[_wm(g,*DAYTIME)][sc].dropna()
            r['pedo_total_steps']=ds.sum(); r['pedo_active_mins']=(ds>0).sum(); r['pedo_peak_steps']=ds.max()
            ns=g[_wm(g,*SLEEP_NIGHT)][sc].dropna()
            r['pedo_night_zero_frac']=(ns==0).mean(); r['pedo_sedentary_hrs']=(ds<5).sum()/60
        if cal: r['pedo_total_calories']=g[cal].sum()
        rows.append(r)
    return pd.DataFrame(rows)

def build_domain_features(raw_dict):
    extractors={'hr':_feat_hr,'screen':_feat_screen,'pedo':_feat_pedo,
                'activity':lambda d:pd.DataFrame(),'light':lambda d:pd.DataFrame()}
    dfs=[]
    for key,fn in extractors.items():
        if key not in raw_dict: continue
        try:
            fdf=fn(raw_dict[key])
            if len(fdf)>0: dfs.append(fdf); print(f'  [domain] {key}: {fdf.shape[1]-2}피처')
        except Exception as e: print(f'  ⚠️ {key}: {e}')
    if not dfs: return pd.DataFrame()
    return reduce(lambda l,r: pd.merge(l,r,on=['subject_id','date'],how='outer'),dfs)

print('Domain feature 함수 정의 완료')

## 3. 레이블 로딩 및 병합

In [ ]:
LABEL_PATH = os.path.join(DATA_DIR, 'ch2026_metrics_train.csv')
labels_df  = pd.read_csv(LABEL_PATH)
labels_df['lifelog_date'] = pd.to_datetime(labels_df['lifelog_date'])
for lc in LABEL_COLS: labels_df[lc] = pd.to_numeric(labels_df[lc], errors='coerce')

merge_labels = labels_df[['subject_id','lifelog_date']+LABEL_COLS].rename(columns={'lifelog_date':'date'})
full_df = pd.merge(feature_df, merge_labels, on=['subject_id','date'], how='inner')
print(f'병합: {full_df.shape}')

# sleep offset deviation (label merge 후)
if 'hr__sleep_offset_hour' in full_df.columns:
    for subj,grp in full_df.groupby('subject_id'):
        full_df.loc[grp.index,'sleep__offset_deviation'] = grp['hr__sleep_offset_hour'] - grp['hr__sleep_offset_hour'].mean()
        full_df.loc[grp.index,'sleep__onset_deviation']  = grp['hr__sleep_onset_hour']  - grp['hr__sleep_onset_hour'].mean()
    full_df = full_df.sort_values(['subject_id','date'])
    full_df['sleep__offset_irregularity'] = (
        full_df.groupby('subject_id')['hr__sleep_offset_hour']
        .transform(lambda x: x.shift(1).rolling(7,min_periods=2).std()))

# rolling z-score deviation + cumulative features
ROLLING_WIN = 14
feat_cols   = [c for c in full_df.columns if c not in ['subject_id','date']+LABEL_COLS]
cum_target  = [c for c in feat_cols if any(k in c.lower() for k in ['pedo','hr','screen']) and c.endswith('__mean')]

domain_df = build_domain_features(raw)

deviation_dfs = []
for subj_id, sdf in full_df.groupby('subject_id'):
    sdf = sdf.sort_values('date').copy()
    num_cols = sdf[feat_cols].select_dtypes(include=[np.number]).columns.tolist()
    roll_mean = sdf[num_cols].shift(1).rolling(ROLLING_WIN,min_periods=3).mean()
    roll_std  = sdf[num_cols].shift(1).rolling(ROLLING_WIN,min_periods=3).std().fillna(1e-6).replace(0,1e-6)
    deviation = (sdf[num_cols]-roll_mean)/roll_std
    deviation.columns=[f'dev__{c}' for c in deviation.columns]
    cum_parts={}
    for w in [3,5]:
        for col in cum_target:
            if col in sdf.columns:
                cum_parts[f'cum{w}__{col}']=sdf[col].shift(1).rolling(w,min_periods=1).sum().values
    result=pd.concat([sdf[['subject_id','date']+LABEL_COLS].reset_index(drop=True),
                      sdf[num_cols].reset_index(drop=True),
                      deviation.reset_index(drop=True),
                      pd.DataFrame(cum_parts,index=sdf.index).reset_index(drop=True)],axis=1)
    deviation_dfs.append(result)

full_dev_df = pd.concat(deviation_dfs,ignore_index=True).dropna(subset=feat_cols[:5])

if len(domain_df)>0:
    full_dev_df=pd.merge(full_dev_df,domain_df,on=['subject_id','date'],how='left')
    domain_feat_cols=[c for c in domain_df.columns if c not in ['subject_id','date']
                      and pd.api.types.is_numeric_dtype(domain_df[c])]
    print(f'도메인 피처 병합: {len(domain_feat_cols)}개')
else:
    domain_feat_cols=[]

print(f'\n✅ 학습 데이터: {full_dev_df.shape}  |  유효 샘플: {len(full_dev_df)}')

## 4. XGBoost 전용 파생 피처 추가
(Lag / Sleep debt / Arousal index / Social jet lag 등)

In [ ]:
def add_xgb_features(df):
    df = df.copy().sort_values(['subject_id','date'])
    g  = df.groupby('subject_id')

    # Lag-1
    for col in ['hr__est_tst_min','hr__sleep_onset_hour','hr__sleep_offset_hour',
                'hr__night_mean','hr__presleep_mean','usage__total_min','usage__evening_min']:
        if col in df.columns: df[f'lag1__{col}'] = g[col].shift(1)

    # Sleep debt
    if 'hr__est_tst_min' in df.columns:
        df['sleep_debt_3d'] = g['hr__est_tst_min'].transform(
            lambda x: (420-x.shift(1)).clip(lower=0).rolling(3,min_periods=1).sum())
        df['sleep_debt_7d'] = g['hr__est_tst_min'].transform(
            lambda x: (420-x.shift(1)).clip(lower=0).rolling(7,min_periods=1).sum())

    # 수면 타이밍 불규칙성
    if 'hr__sleep_onset_hour' in df.columns:
        df['sleep_onset_regularity_7d'] = g['hr__sleep_onset_hour'].transform(
            lambda x: x.shift(1).rolling(7,min_periods=3).std())

    # Pre-sleep arousal index
    parts=[]
    if 'hr__presleep_mean' in df.columns and 'hr__mean' in df.columns:
        exc=(df['hr__presleep_mean']-g['hr__mean'].transform('mean')).clip(lower=0)
        parts.append(exc/(g['hr__mean'].transform('std').replace(0,1))*0.4)
    if 'screen_presleep_on_frac' in df.columns:
        parts.append(df['screen_presleep_on_frac'].fillna(0)*0.35)
    if 'usage__evening_ratio' in df.columns:
        parts.append(df['usage__evening_ratio'].fillna(0)*0.25)
    if parts: df['presleep_arousal_index']=sum(parts)

    # Activity-recovery ratio
    if 'hr__resting' in df.columns:
        df['activity_recovery_ratio']=df['hr__resting'].fillna(60)/g['hr__resting'].transform('mean').replace(0,60)

    # 연속 수면 부족 streak
    if 'hr__est_tst_min' in df.columns:
        is_short=(df['hr__est_tst_min'].shift(1)<g['hr__est_tst_min'].transform('mean')).astype(int)
        def streak(s):
            cnt,res=[],[]
            for v in s:
                cnt.append((cnt[-1]+1 if cnt else 1) if v==1 else 0)
            return cnt
        df['low_sleep_streak']=df.assign(_s=is_short).groupby('subject_id')['_s'].transform(streak)

    # Social jet lag
    if 'hr__sleep_onset_hour' in df.columns and 'time__is_weekend' in df.columns:
        jl={}
        for subj,sdf in df.groupby('subject_id'):
            we=sdf[sdf['time__is_weekend']==1]['hr__sleep_onset_hour'].mean()
            wd=sdf[sdf['time__is_weekend']==0]['hr__sleep_onset_hour'].mean()
            jl[subj]=abs(we-wd) if not (np.isnan(we) or np.isnan(wd)) else 0.0
        df['social_jetlag']=df['subject_id'].map(jl)

    # Screen-sleep delay
    if 'screen_last_off_hour' in df.columns and 'hr__sleep_onset_hour' in df.columns:
        df['screen_sleep_delay']=(df['screen_last_off_hour'].fillna(22)*
                                   df['hr__sleep_onset_hour'].fillna(23)/(22*23))

    new=[c for c in df.columns if c not in full_dev_df.columns]
    print(f'✅ XGB 파생 피처 {len(new)}개 추가')
    return df

xgb_df = add_xgb_features(full_dev_df)

## 5. 피처 정제 (결측 제거 → 저분산 제거 → 고상관 제거 → QuantileTransform)

In [ ]:
def refine_features(df, label_cols, missing_thresh=0.5, var_thresh=1e-4, corr_thresh=0.97):
    exclude={'subject_id','date'}|set(label_cols)
    raw_candidates=[c for c in df.columns if c not in exclude
                    and not c.startswith('dev__') and not c.startswith('cum')
                    and pd.api.types.is_numeric_dtype(df[c])]
    print(f'후보 피처: {len(raw_candidates)}개')

    missing_rate=df[raw_candidates].isna().mean()
    keep=missing_rate[missing_rate<=missing_thresh].index.tolist()
    print(f'  결측률>{missing_thresh:.0%} 제거: {len(raw_candidates)-len(keep)}개 → {len(keep)}개')

    df_imp=df[['subject_id','date']+list(label_cols)+keep].copy()
    for col in keep:
        df_imp[col]=df_imp[col].fillna(df_imp.groupby('subject_id')[col].transform('median')).fillna(df_imp[col].median())

    variances=df_imp[keep].var()
    keep2=variances[variances>var_thresh].index.tolist()
    print(f'  저분산 제거: {len(keep)-len(keep2)}개 → {len(keep2)}개')

    corr_m=df_imp[keep2].corr().abs()
    upper=corr_m.where(np.triu(np.ones(corr_m.shape),k=1).astype(bool))
    drop_c=[c for c in upper.columns if any(upper[c]>corr_thresh)]
    keep3=[c for c in keep2 if c not in drop_c]
    print(f'  고상관>{corr_thresh} 제거: {len(drop_c)}개 → {len(keep3)}개')

    qt=QuantileTransformer(output_distribution='normal',random_state=42)
    df_imp[keep3]=qt.fit_transform(df_imp[keep3])
    print(f'\n✅ 최종 피처: {len(keep3)}개')
    return df_imp, keep3, qt

xgb_refined, feat_cols_xgb, qt = refine_features(xgb_df, LABEL_COLS)
print(f'학습 데이터: {xgb_refined.shape}')

## 6. XGBoost LOSO-CV

In [ ]:
def find_best_thr(y_true, y_prob):
    mask=~np.isnan(y_true)
    if mask.sum()<4: return 0.5
    best_t,best_f1=0.5,0.0
    for t in np.arange(0.25,0.76,0.02):
        f1=f1_score(y_true[mask],(y_prob[mask]>=t).astype(int),zero_division=0)
        if f1>best_f1: best_f1,best_t=f1,t
    return round(best_t,2)

XGB_PARAMS = dict(n_estimators=500, max_depth=4, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                  reg_alpha=0.1, reg_lambda=1.0, eval_metric='auc',
                  use_label_encoder=False, random_state=42, verbosity=0)

all_subjects = sorted(xgb_refined['subject_id'].unique())
xgb_loso_results = {}

for test_subj in all_subjects:
    print(f'\n{"="*50}\nTest: {test_subj}\n{"="*50}')
    train_df=xgb_refined[xgb_refined['subject_id']!=test_subj]
    test_df =xgb_refined[xgb_refined['subject_id']==test_subj]
    X_train=train_df[feat_cols_xgb].values; X_test=test_df[feat_cols_xgb].values
    subj_metrics={}; subj_imp={}

    for lbl in LABEL_COLS:
        y_train=train_df[lbl].values; y_test=test_df[lbl].values
        tr_mask=~np.isnan(y_train); te_mask=~np.isnan(y_test)
        if tr_mask.sum()<10 or te_mask.sum()<2: continue
        pos_rate=y_train[tr_mask].mean()
        model=xgb.XGBClassifier(**XGB_PARAMS,
                                 scale_pos_weight=(1-pos_rate)/(pos_rate+1e-6),
                                 early_stopping_rounds=30)
        n_val=max(1,int(tr_mask.sum()*0.2))
        val_idx=np.where(tr_mask)[0][-n_val:]; tr_idx=np.where(tr_mask)[0][:-n_val]
        model.fit(X_train[tr_idx],y_train[tr_idx],
                  eval_set=[(X_train[val_idx],y_train[val_idx])],verbose=False)
        y_prob=model.predict_proba(X_test[te_mask])[:,1]; y_true=y_test[te_mask]
        thr=find_best_thr(y_true,y_prob); y_hat=(y_prob>=thr).astype(int)
        try: auc=roc_auc_score(y_true,y_prob)
        except: auc=0.5
        subj_metrics[lbl]={'acc':accuracy_score(y_true,y_hat),
                            'f1':f1_score(y_true,y_hat,zero_division=0),'auc':auc,'thr':thr}
        subj_imp[lbl]=model.feature_importances_
        print(f'  {lbl}: AUC={auc:.3f}  F1={subj_metrics[lbl]["f1"]:.3f}  thr={thr:.2f}')

    if subj_metrics:
        subj_metrics['macro']={k:np.mean([v[k] for v in subj_metrics.values() if k in v])
                                for k in ['acc','f1','auc']}
        print(f'  macro AUC={subj_metrics["macro"]["auc"]:.3f}')
    xgb_loso_results[test_subj]={'metrics':subj_metrics,'importance':subj_imp}

print('\n✅ LOSO-CV 완료!')

## 7. 결과 요약

In [ ]:
xgb_summary={lbl:{'acc':[],'f1':[],'auc':[]} for lbl in LABEL_COLS+['macro']}
for subj,res in xgb_loso_results.items():
    for lbl,m in res['metrics'].items():
        if lbl in xgb_summary:
            for k in ['acc','f1','auc']: xgb_summary[lbl][k].append(m.get(k,0))

print('XGBoost LOSO-CV 결과')
print(f'{"레이블":8s}  {"Acc":>12s}  {"F1":>12s}  {"AUC":>12s}')
print('-'*52)
for lbl in LABEL_COLS+['macro']:
    if not xgb_summary[lbl]['auc']: continue
    am=np.mean(xgb_summary[lbl]['acc']); as_=np.std(xgb_summary[lbl]['acc'])
    fm=np.mean(xgb_summary[lbl]['f1']);  fs=np.std(xgb_summary[lbl]['f1'])
    um=np.mean(xgb_summary[lbl]['auc']); us=np.std(xgb_summary[lbl]['auc'])
    if lbl=='macro': print('─'*52)
    print(f'{lbl:8s}  {am:.3f}±{as_:.3f}  {fm:.3f}±{fs:.3f}  {um:.3f}±{us:.3f}')

## 8. 피처 중요도

In [ ]:
avg_imp={lbl:np.zeros(len(feat_cols_xgb)) for lbl in LABEL_COLS}
cnt={lbl:0 for lbl in LABEL_COLS}
for subj,res in xgb_loso_results.items():
    for lbl,imp in res['importance'].items():
        if lbl in avg_imp: avg_imp[lbl]+=imp; cnt[lbl]+=1
for lbl in LABEL_COLS:
    if cnt[lbl]>0: avg_imp[lbl]/=cnt[lbl]

fig,axes=plt.subplots(2,4,figsize=(20,10)); axes=axes.flatten()
for ax_idx,lbl in enumerate(LABEL_COLS):
    ax=axes[ax_idx]; imp=avg_imp[lbl]
    if imp.sum()==0: ax.set_title(f'{lbl} (없음)'); continue
    top=np.argsort(imp)[-15:][::-1]
    feats=[feat_cols_xgb[i][:30] for i in top]
    bars=ax.barh(range(15),imp[top][::-1],color='#4C9BE8' if lbl in Q_COLS else '#E87C4C',alpha=0.8)
    ax.set_yticks(range(15)); ax.set_yticklabels(feats[::-1],fontsize=7)
    ax.set_title(f'{lbl} Top-15',fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('Feature Importance by Label',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()

all_avg=np.mean([avg_imp[l] for l in LABEL_COLS],axis=0)
top20=np.argsort(all_avg)[-20:][::-1]
print('\n전체 평균 중요도 Top-20:')
for rank,i in enumerate(top20,1):
    print(f'  {rank:2d}. {feat_cols_xgb[i]:50s}  {all_avg[i]:.4f}')

## 9. 최종 모델 재학습 및 저장

In [ ]:
import joblib
final_xgb_models={}; final_thresholds={}
X_all=xgb_refined[feat_cols_xgb].values

for lbl in LABEL_COLS:
    y_all=xgb_refined[lbl].values; mask=~np.isnan(y_all)
    if mask.sum()<10: continue
    pos_rate=y_all[mask].mean()
    model=xgb.XGBClassifier(**{k:v for k,v in XGB_PARAMS.items() if k!='eval_metric'},
                             scale_pos_weight=(1-pos_rate)/(pos_rate+1e-6),
                             n_estimators=300,verbosity=0)
    model.fit(X_all[mask],y_all[mask])
    final_xgb_models[lbl]=model
    thr_vals=[xgb_loso_results[s]['metrics'].get(lbl,{}).get('thr',0.5) for s in xgb_loso_results]
    final_thresholds[lbl]=round(np.mean(thr_vals),2)
    print(f'  {lbl}: thr={final_thresholds[lbl]:.2f}')

joblib.dump({'models':final_xgb_models,'thresholds':final_thresholds,
             'feat_cols':feat_cols_xgb,'qt':qt},'/content/xgb_models.pkl')
print('\n✅ 저장: /content/xgb_models.pkl')

## 10. 제출 파일 생성

In [ ]:
sub=pd.read_csv('/content/drive/MyDrive/ch2025/ch2026_submission_sample.csv')
sub['lifelog_date']=pd.to_datetime(sub['lifelog_date'])
sub['sleep_date']=pd.to_datetime(sub['sleep_date'])

# feature_df는 위에서 이미 빌드됨 (모든 날짜 포함)
feature_df['date']=pd.to_datetime(feature_df['date'])
full_feat=add_xgb_features(feature_df)   # 테스트 날짜에도 동일 파생 피처 적용

test_feat=pd.merge(sub[['subject_id','sleep_date','lifelog_date']],
                   full_feat.rename(columns={'date':'lifelog_date'}),
                   on=['subject_id','lifelog_date'],how='left')

for col in feat_cols_xgb:
    if col not in test_feat.columns: test_feat[col]=np.nan

subj_med=xgb_refined.groupby('subject_id')[feat_cols_xgb].median()
glob_med=xgb_refined[feat_cols_xgb].median()
for col in feat_cols_xgb:
    for subj in test_feat['subject_id'].unique():
        mask=test_feat['subject_id']==subj
        if subj in subj_med.index:
            test_feat.loc[mask,col]=test_feat.loc[mask,col].fillna(subj_med.loc[subj,col])
    test_feat[col]=test_feat[col].fillna(glob_med[col])

X_test=qt.transform(test_feat[feat_cols_xgb].values)

for lbl in LABEL_COLS:
    if lbl not in final_xgb_models: sub[lbl]=0; continue
    prob=final_xgb_models[lbl].predict_proba(X_test)[:,1]
    sub[lbl]=(prob>=final_thresholds.get(lbl,0.5)).astype(int)

out=sub[['subject_id','sleep_date','lifelog_date']+LABEL_COLS]
out.to_csv('/content/submission.csv',index=False)
print(f'✅ 저장: /content/submission.csv  ({len(out)}행)')
print('\n예측 분포 (1의 비율):')
print(out[LABEL_COLS].mean().round(3).to_string())
out.head()

In [ ]:
from google.colab import files
files.download('/content/submission.csv')